# 04 – GARCH models

Compare GARCH(1,1), GJR-GARCH and EGARCH using the same out-of-sample dates as the HAR-RV model.

In [1]:
import numpy as np
import pandas as pd

from arch import arch_model
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Prepare daily intraday returns

In [2]:
raw = pd.read_csv("../data/raw/spy_5min_2024-10-03_2026-09-30.csv")

raw["datetime"] = pd.to_datetime(
    raw["datetime"], utc=True
).dt.tz_convert("America/New_York")

regular = raw[
    (raw["datetime"].dt.time >= pd.to_datetime("09:30").time())
    & (raw["datetime"].dt.time < pd.to_datetime("16:00").time())
].copy()

regular["date"] = pd.to_datetime(regular["datetime"].dt.date)

rv_data = pd.read_csv(
    "../data/spy_daily_realized_volatility.csv",
    parse_dates=["date"],
)

valid_dates = set(rv_data["date"])

daily = (
    regular[regular["date"].isin(valid_dates)]
    .groupby("date")
    .agg(open_price=("o", "first"), close_price=("c", "last"))
    .reset_index()
)

# Percentage returns are numerically more convenient for arch.
daily["return"] = np.log(daily["close_price"] / daily["open_price"]) * 100

daily.head()

,date,open_price,close_price,return
0,2024-10-04,572.35,572.91,0.097794
1,2024-10-07,571.30,567.81,-0.612761
2,2024-10-08,570.42,573.13,0.473964
3,2024-10-09,573.16,577.10,0.685065
4,2024-10-10,575.77,576.10,0.057298


In [3]:
har_forecasts = pd.read_csv(
    "../results/har_rv_forecasts.csv",
    parse_dates=["target_date"],
)

first_test_date = har_forecasts["target_date"].min()
train_returns = daily.loc[daily["date"] < first_test_date, "return"]

print("Training observations:", len(train_returns))
print("Test observations:", len(har_forecasts))

Training observations: 398
Test observations: 95


## In-sample model summaries

In [4]:
garch_fit = arch_model(
    train_returns, mean="Constant", vol="GARCH", p=1, q=1, dist="normal"
).fit(disp="off")

garch_fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -433.998
Distribution:                  Normal   AIC:                           875.996
Method:            Maximum Likelihood   BIC:                           891.942
                                        No. Observations:                  398
Date:                Sun, Oct 04 2026   Df Residuals:                      397
Time:                        13:19:51   Df Model:                            1
                                  Mean Model                                 
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
mu             0.0254  3.164e-02      0.804      0.421 [-3.657e-02,8.744e-02]
                             Volatility Model                             
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega          0.1191  5.104e-02      2.333  1.966e-02 [1.902e-02,  0.219]
alpha[1]       0.2122  6.439e-02      3.296  9.818e-04 [8.601e-02,  0.338]
beta[1]        0.5887  9.270e-02      6.351  2.138e-10   [  0.407,  0.770]
==========================================================================

Covariance estimator: robust
"""

In [5]:
gjr_fit = arch_model(
    train_returns, mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="normal"
).fit(disp="off")

gjr_fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                   Constant Mean - GJR-GARCH Model Results                    
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                  GJR-GARCH   Log-Likelihood:               -426.891
Distribution:                  Normal   AIC:                           863.782
Method:            Maximum Likelihood   BIC:                           883.714
                                        No. Observations:                  398
Date:                Sun, Oct 04 2026   Df Residuals:                      397
Time:                        13:19:51   Df Model:                            1
                                  Mean Model                                  
==============================================================================
                  coef    std err          t      P>|t|       95.0% Conf. Int.
------------------------------------------------------------------------------
mu         -1.6555e-03  3.409e-02 -4.857e-02      0.961 [-6.847e-02,6.516e-02]
                             Volatility Model                             
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega          0.1268  4.883e-02      2.597  9.407e-03 [3.110e-02,  0.223]
alpha[1]       0.0000  9.035e-02      0.000      1.000   [ -0.177,  0.177]
gamma[1]       0.4233      0.192      2.202  2.764e-02 [4.660e-02,  0.800]
beta[1]        0.5784      0.112      5.171  2.327e-07   [  0.359,  0.798]
==========================================================================

Covariance estimator: robust
"""

In [6]:
egarch_fit = arch_model(
    train_returns, mean="Constant", vol="EGARCH", p=1, o=1, q=1, dist="normal"
).fit(disp="off")

egarch_fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - EGARCH Model Results                     
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -427.799
Distribution:                  Normal   AIC:                           865.597
Method:            Maximum Likelihood   BIC:                           885.529
                                        No. Observations:                  398
Date:                Sun, Oct 04 2026   Df Residuals:                      397
Time:                        13:19:52   Df Model:                            1
                                  Mean Model                                  
==============================================================================
                  coef    std err          t      P>|t|       95.0% Conf. Int.
------------------------------------------------------------------------------
mu         -2.8108e-03  3.401e-02 -8.265e-02      0.934 [-6.946e-02,6.384e-02]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega         -0.1151  5.188e-02     -2.218  2.657e-02 [ -0.217,-1.337e-02]
alpha[1]       0.2712  8.110e-02      3.345  8.239e-04    [  0.112,  0.430]
gamma[1]      -0.2657      0.100     -2.649  8.080e-03 [ -0.462,-6.910e-02]
beta[1]        0.8108  7.400e-02     10.958  6.085e-28    [  0.666,  0.956]
===========================================================================

Covariance estimator: robust
"""

## Expanding-window forecasts

In [7]:
def expanding_garch_forecast(vol="GARCH", o=0):
    forecasts = []

    for target_date in har_forecasts["target_date"]:
        train = daily.loc[daily["date"] < target_date, "return"]

        model = arch_model(
            train,
            mean="Constant",
            vol=vol,
            p=1,
            o=o,
            q=1,
            dist="normal",
        )

        fit = model.fit(disp="off")
        variance_pct = fit.forecast(horizon=1).variance.iloc[-1, 0]
        forecasts.append(variance_pct / 10000)

    return np.array(forecasts)


har_forecasts["garch_rv_forecast"] = expanding_garch_forecast("GARCH", o=0)
har_forecasts["gjr_rv_forecast"] = expanding_garch_forecast("GARCH", o=1)
har_forecasts["egarch_rv_forecast"] = expanding_garch_forecast("EGARCH", o=1)

## Compare forecast accuracy

In [8]:
def qlike_loss(actual, forecast):
    ratio = actual / forecast
    return ratio - np.log(ratio) - 1


def evaluate_model(name, forecast_col):
    log_forecast = np.log(har_forecasts[forecast_col])

    return {
        "Model": name,
        "MAE": mean_absolute_error(har_forecasts["target"], log_forecast),
        "RMSE": np.sqrt(mean_squared_error(har_forecasts["target"], log_forecast)),
        "QLIKE": qlike_loss(
            har_forecasts["actual_rv"], har_forecasts[forecast_col]
        ).mean(),
    }


garch_results = pd.DataFrame([
    evaluate_model("GARCH(1,1)", "garch_rv_forecast"),
    evaluate_model("GJR-GARCH(1,1)", "gjr_rv_forecast"),
    evaluate_model("EGARCH(1,1)", "egarch_rv_forecast"),
])

har_results = pd.read_csv("../results/har_rv_results.csv")
comparison = pd.concat([har_results, garch_results], ignore_index=True)

comparison

,Model,MAE,RMSE,QLIKE
0,HAR-RV,0.474115,0.643004,0.236613
1,Naive,0.533653,0.723066,0.302232
2,"GARCH(1,1)",0.705010,0.818413,0.271446
3,"GJR-GARCH(1,1)",0.691317,0.808089,0.253642
4,"EGARCH(1,1)",0.693454,0.827625,0.259192


## Save results

In [9]:
comparison.to_csv("../results/garch_model_comparison.csv", index=False)

har_forecasts[[
    "target_date",
    "actual_rv",
    "garch_rv_forecast",
    "gjr_rv_forecast",
    "egarch_rv_forecast",
]].to_csv("../results/garch_forecasts.csv", index=False)